In [ ]:
from transformers import pipeline

In [1]:
!pip install gitpython




In [2]:
!git config --global user.email "viveksolanki.17122002@gmail.com"
!git config --global user.name "vvek17"



In [4]:
!git clone https://github.com/vvek17/A-Comparative-Study-of-Faithfulness-Detection-Methods-.git
%cd A-Comparative-Study-of-Faithfulness-Detection-Methods-

Cloning into 'A-Comparative-Study-of-Faithfulness-Detection-Methods-'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), done.
/content/A-Comparative-Study-of-Faithfulness-Detection-Methods-


In [ ]:
!git add .
!git commit -m "Add Python scraping scripts"
!git push https://vvek17:<YOUR_GITHUB_TOKEN>@github.com/vvek17/DimondKG-Project.git

** *italicized text*model="facebook/bart-large-mnli",**


In [ ]:

from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# Model from Meta
classifier = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    return_all_scores=True
)

dataset = load_dataset("pminervini/HaluEval", "qa_samples")

all_predictions = []
all_true_labels = []

print("Running inference on 10000 samples...")

for i in range(10000):
    knowledge = dataset['data'][i]['knowledge']
    answer = dataset['data'][i]['answer']
    true_label = dataset['data'][i]['hallucination']

    # NLI format
    input_text = f"{knowledge} </s> {answer}"

    result = classifier(input_text)

    # Debug: print structure of first result
    if i == 0:
        print("Result structure:")
        print(result)
        print("Type:", type(result))
        if isinstance(result, list) and len(result) > 0:
            print("First element:", result[0])
            print("Type of first element:", type(result[0]))

    # Get highest scoring label
    # Result structure: [[{'label': 'CONTRADICTION', 'score': 0.9}, {'label': 'NEUTRAL', 'score': 0.05}, ...]]
    scores = result[0] if isinstance(result[0], list) else result
    best_label = max(scores, key=lambda x: x['score'])['label']

    # Map prediction
    if best_label.upper() == "CONTRADICTION":
        prediction = "yes"
    else:
        prediction = "no"

    all_predictions.append(prediction)
    all_true_labels.append(true_label)

# METRICS CALCULATION
print("\n" + "="*70)
print("EVALUATION METRICS")
print("="*70)

# Accuracy
accuracy = sum(p == t for p, t in zip(all_predictions, all_true_labels)) / 10000
print(f"\nAccuracy: {accuracy:.4f}")

# Sklearn metrics
precision = precision_score(all_true_labels, all_predictions, pos_label="yes")
recall = recall_score(all_true_labels, all_predictions, pos_label="yes")
f1 = f1_score(all_true_labels, all_predictions, pos_label="yes")

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT")
print("="*70)
print(classification_report(all_true_labels, all_predictions))

# ADDITIONAL INSIGHTS
print("\n" + "="*70)
print("PREDICTION DISTRIBUTION")
print("="*70)
yes_count = all_predictions.count("yes")
no_count = all_predictions.count("no")
print(f"Predicted 'yes': {yes_count} ({yes_count/10000*100:.2f}%)")
print(f"Predicted 'no': {no_count} ({no_count/10000*100:.2f}%)")

true_yes_count = all_true_labels.count("yes")
true_no_count = all_true_labels.count("no")
print(f"\nTrue 'yes': {true_yes_count} ({true_yes_count/10000*100:.2f}%)")
print(f"True 'no': {true_no_count} ({true_no_count/10000*100:.2f}%)")

print("\nEvaluation complete!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

qa_samples/data-00000-of-00001.parquet:   0%|          | 0.00/3.43M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Running inference on 10000 samples...
Result structure:
[{'label': 'contradiction', 'score': 0.508694052696228}]
Type: <class 'list'>
First element: {'label': 'contradiction', 'score': 0.508694052696228}
Type of first element: <class 'dict'>


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



EVALUATION METRICS

Accuracy: 0.6876
Precision: 0.7296
Recall: 0.5982
F1: 0.6574

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

          no       0.66      0.78      0.71      4990
         yes       0.73      0.60      0.66      5010

    accuracy                           0.69     10000
   macro avg       0.69      0.69      0.69     10000
weighted avg       0.69      0.69      0.69     10000


PREDICTION DISTRIBUTION
Predicted 'yes': 4108 (41.08%)
Predicted 'no': 5892 (58.92%)

True 'yes': 5010 (50.10%)
True 'no': 4990 (49.90%)

Evaluation complete!


**model="facebook/bart-large-mnli", with Summac-C Logic**


In [ ]:
# 1. Install necessary libraries
!pip install -q transformers datasets scikit-learn tqdm

import torch
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm

# 2. Setup Device (GPU is essential for HaluEval's size)
device = 0 if torch.cuda.is_available() else -1

# 3. Initialize the NLI Pipeline
classifier = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=device,
    return_all_scores=True # Added to ensure the pipeline returns all scores as a list of dictionaries
)

# 4. Load HaluEval QA Samples
dataset = load_dataset("pminervini/HaluEval", "qa_samples", split='data')

all_predictions = []
all_true_labels = []

# Using 10000 for full run
limit = 10000

print(f"Starting Evaluation on {limit} samples...")

for i in tqdm(range(limit)):
    # Correct indexing for HaluEval
    item = dataset[i]
    knowledge = item['knowledge']
    answer = item['answer']
    true_label = item['hallucination'] # "yes" or "no"

    # Call the classifier
    classification_results = classifier({"text": knowledge, "text_pair": answer})

    # Normalize classification_results to always be a list of dictionaries for iteration

    if isinstance(classification_results, list) and len(classification_results) > 0:
        if isinstance(classification_results[0], list):
            # Case 1: List of lists, take the inner list
            scores_list = classification_results[0]
        else:
            # Case 2: Already a list of dictionaries
            scores_list = classification_results
    elif isinstance(classification_results, dict):
        # Case 3: Single dictionary, wrap it in a list
        scores_list = [classification_results]
    else:
        # Fallback for any other unexpected format
        # This should ideally not happen if return_all_scores=True is reliable
        scores_list = [] # Or raise an error

    scores = {res['label'].lower(): res['score'] for res in scores_list}

    # SummaC-Zero logic: Contradiction > Entailment = Hallucination
    if scores.get('contradiction', 0) > scores.get('entailment', 0):
        prediction = "yes"
    else:
        prediction = "no"

    all_predictions.append(prediction)
    all_true_labels.append(true_label)

# 5. Metrics Calculation
print("\n" + "="*30)
print("--- SUMMAC-ZERO RESULTS ---")
print("="*30)

accuracy = sum(p == t for p, t in zip(all_predictions, all_true_labels)) / len(all_true_labels)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision_score(all_true_labels, all_predictions, pos_label='yes'):.4f}")
print(f"Recall:    {recall_score(all_true_labels, all_predictions, pos_label='yes'):.4f}")
print(f"F1 Score:  {f1_score(all_true_labels, all_predictions, pos_label='yes'):.4f}")

print("\nDetailed Classification Report:")
print(classification_report(all_true_labels, all_predictions))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Starting Evaluation on 10000 samples...


100%|██████████| 10000/10000 [06:05<00:00, 27.33it/s]



--- SUMMAC-ZERO RESULTS ---
Accuracy: 0.6722
Precision: 0.7085
Recall:    0.5874
F1 Score:  0.6423

Detailed Classification Report:
              precision    recall  f1-score   support

          no       0.65      0.76      0.70      4990
         yes       0.71      0.59      0.64      5010

    accuracy                           0.67     10000
   macro avg       0.68      0.67      0.67     10000
weighted avg       0.68      0.67      0.67     10000



**MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"**

In [ ]:
import torch
from transformers import pipeline
from datasets import load_dataset
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import Hugging Face login function
from huggingface_hub import login

# CONFIGURATION

MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
DATASET_NAME = "pminervini/HaluEval"
DATASET_CONFIG = "qa_samples"
NUM_SAMPLES = 10000
BATCH_SIZE = 16  # Optimized for GPU memory; adjust 8-32 based on your GPU
CANDIDATE_LABELS = ["yes", "no"]

# Authenticate with Hugging Face (if needed)
# Uncomment the line below and run the cell if you face authentication issues.
# login()

# MODEL INITIALIZATION
print("Initializing model...")
device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU (CUDA)' if device == 0 else 'CPU'}")

classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    device=device
)

# DATASET LOADING
print(f"\nLoading dataset: {DATASET_NAME}/{DATASET_CONFIG}...")
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)

# Take first 10,000 samples from data split
eval_data = dataset['data'].select(range(NUM_SAMPLES))
print(f"Loaded {len(eval_data)} samples")

# DATA PREPARATION
print("\nPreparing data for batch inference...")

# Extract texts and true labels
texts = []
true_labels = []

for sample in eval_data:
    # Construct text for classification: Question + Answer
    text = f"Question: {sample['question']} Answer: {sample['answer']}"
    texts.append(text)

    # True label is the 'hallucination' field
    true_labels.append(sample['hallucination'])

print(f"Prepared {len(texts)} text samples")
print(f"Sample text: {texts[0][:100]}...")
print(f"Sample label: {true_labels[0]}")

# BATCH INFERENCE
print(f"\nRunning batch inference (batch_size={BATCH_SIZE})...")

all_predictions = []

# Process in batches with progress bar
num_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE

with torch.no_grad():  # Disable gradient computation for inference
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Inference", total=num_batches):
        batch_texts = texts[i:i + BATCH_SIZE]

        # Batch inference - single call for multiple texts
        results = classifier(
            batch_texts,
            candidate_labels=CANDIDATE_LABELS,
            multi_label=False  # Binary classification
        )

        # Extract predictions from batch results
        if isinstance(results, list):
            # Multiple samples - results is a list of dicts
            for result in results:
                predicted_label = result['labels'][0]  # Top label
                all_predictions.append(predicted_label)
        else:
            # Single sample - results is a dict
            predicted_label = results['labels'][0]
            all_predictions.append(predicted_label)

print(f"Inference complete! Processed {len(all_predictions)} samples")

# VALIDATION
assert len(all_predictions) == NUM_SAMPLES, "Prediction count mismatch!"
assert len(true_labels) == NUM_SAMPLES, "True label count mismatch!"
print(f"Validation passed: {NUM_SAMPLES} predictions generated")

# METRICS CALCULATION
print("\n" + "="*70)
print("EVALUATION METRICS")
print("="*70)

# Accuracy
accuracy = sum(p == t for p, t in zip(all_predictions, true_labels)) / NUM_SAMPLES
print(f"\nAccuracy: {accuracy:.4f}")

# Sklearn metrics
precision = precision_score(true_labels, all_predictions, pos_label="yes")
recall = recall_score(true_labels, all_predictions, pos_label="yes")
f1 = f1_score(true_labels, all_predictions, pos_label="yes")

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT")
print("="*70)
print(classification_report(true_labels, all_predictions))

# ADDITIONAL INSIGHTS
print("\n" + "="*70)
print("PREDICTION DISTRIBUTION")
print("="*70)
yes_count = all_predictions.count("yes")
no_count = all_predictions.count("no")
print(f"Predicted 'yes': {yes_count} ({yes_count/NUM_SAMPLES*100:.2f}%)")
print(f"Predicted 'no': {no_count} ({no_count/NUM_SAMPLES*100:.2f}%)")

true_yes_count = true_labels.count("yes")
true_no_count = true_labels.count("no")
print(f"\nTrue 'yes': {true_yes_count} ({true_yes_count/NUM_SAMPLES*100:.2f}%)")
print(f"True 'no': {true_no_count} ({true_no_count/NUM_SAMPLES*100:.2f}%)")

print("\nEvaluation complete!")

Initializing model...
Device: CPU


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Loading dataset: pminervini/HaluEval/qa_samples...
Loaded 10000 samples

Preparing data for batch inference...
Prepared 10000 text samples
Sample text: Question: Which magazine was started first Arthur's Magazine or First for Women? Answer: First for W...
Sample label: yes

Running batch inference (batch_size=16)...


Inference:   0%|          | 1/625 [05:54<61:31:10, 354.92s/it]


KeyboardInterrupt: 

In [ ]:
!pip install --upgrade transformers huggingface_hub


  Using cached huggingface_hub-1.10.2-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-1.10.2-py3-none-any.whl (642 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.16.4
    Uninstalling huggingface-hub-0.16.4:
      Successfully uninstalled huggingface-hub-0.16.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.


For Preprocess, data is already lable, so there is barely preprocessing.

i am going to merge Knowladge and question in one string and another would be a answer. and split dataset in Train/validation/split.

https://huggingface.co/docs/datasets/process (written already how you should make one row with use of  .map())

I did have to merge all the data into one input becasue the model can takes only one input and give us one predicted out.



In [ ]:
from transformers import pipeline
from datasets import load_dataset
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding





# Load the dataset (only once)
dataset = load_dataset("pminervini/HaluEval", "qa_samples", split = 'data')



# new fields:
# input_text → your X (what goes into DeBERTa) with use of .map
# label → your Y (what the model learns to predict)

def prepare_input_text(example):
  # Access fields within the example dictionary for each row
  example['input_text'] = f"Documentation: {example['knowledge']} Question: {example['question']} Answer: {example['answer']}"
  example['label'] = 1 if example['hallucination'] == 'yes' else 0
  return example

# Apply the function to create the new 'input_text' column for each split (e.g., 'data')
dataset = dataset.map(prepare_input_text)

print(dataset[0]['input_text'])
print(dataset[0]['label'])

#test and training split
split_dataset = dataset.train_test_split(test_size=0.3, seed = 42 )
train_dataset = split_dataset['train']
temp_dataset = split_dataset['test']

#now temp_dataset into two split: validation and test

split_dataset = temp_dataset.train_test_split(test_size = 0.5, seed = 42)
validation_dataset = split_dataset['train']
test_dataset = split_dataset['test']
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small", use_fast=True)

#tokenation function

def tokenizer_function(example):
      return tokenizer (
          example['input_text'],
          truncation=True,
          padding = "max_length",
          max_length = 256
      )

# using this tokenization function for train,validation test seperataly so its not gonna be data leak

train_dataset = train_dataset.map(tokenizer_function, batched=True)
validation_dataset = validation_dataset.map(tokenizer_function, batched=True)
test_dataset = test_dataset.map(tokenizer_function, batched=True)

print(train_dataset[0]["input_ids"])
print(train_dataset[0]["attention_mask"])


README.md: 0.00B [00:00, ?B/s]

qa_samples/data-00000-of-00001.parquet:   0%|          | 0.00/3.43M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Documentation: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA. Question: Which magazine was started first Arthur's Magazine or First for Women? Answer: First for Women was started first.
1


...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

[28509, 294, 855, 6274, 680, 16313, 270, 405, 496, 311, 7968, 283, 5780, 649, 280, 268, 307, 476, 4405, 264, 6252, 458, 1961, 273, 268, 309, 263, 307, 476, 1310, 280, 297, 4405, 264, 3431, 4031, 367, 309, 287, 2637, 313, 327, 1920, 285, 261, 262, 8392, 48305, 1898, 261, 307, 1502, 13401, 325, 2745, 309, 263, 279, 75297, 268, 280, 1710, 261, 307, 115459, 23245, 260, 309, 1502, 13401, 325, 2745, 269, 262, 35607, 2998, 1898, 293, 733, 657, 2174, 1704, 261, 19209, 260, 7581, 294, 463, 371, 283, 262, 75297, 268, 1139, 25607, 23245, 261, 4279, 43769, 16583, 284, 4834, 265, 307, 1502, 13401, 325, 2745, 261, 309, 262, 35607, 1898, 293, 319, 2174, 1704, 302, 10519, 294, 4279, 43769, 16583, 284, 262, 4834, 265, 262, 75297, 268, 1139, 25607, 23245, 263, 19209, 280, 268, 38886, 1898, 260, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

Step 1 — Loading the model


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=2
)



Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

lm_predictions.lm_head,
mask_predictions

These are weights from the original pre-training task (predicting masked words). You don't need them for classification so they get dropped.


**Missing Class**
pooler.dense.bias
classifier.weight
pooler.dense.weight
classifier.bias

These are your NEW classification head weights that don't exist in the pre-trained model yet. They get randomly initialized and will be learned during fine-tuning.



Step 2 - Optimizer

In [ ]:
from torch.optim import AdamW
# AdamW is generally preferred for transformers for better generalization
optimizer = AdamW(model.parameters(), lr=5e-5)


Step 3 — Training Arguments




In [ ]:
!pip install evaluate

In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification
import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])
validation_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=2
)

# Compute metrics
def compute_metrics(eval_pred):
    f1_metric = evaluate.load("f1")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"]
    precision = precision_metric.compute(predictions=predictions, references=labels, average="binary")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="binary")["recall"]
    return {"f1": f1, "precision": precision, "recall": recall}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_dir="./logs",
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

trainer.train()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.000000,nan,0.000000,0.000000,0.000000
2,0.000000,nan,0.000000,0.000000,0.000000
3,0.000000,nan,0.000000,0.000000,0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1314, training_loss=0.09336362690686091, metrics={'train_runtime': 301.5906, 'train_samples_per_second': 69.631, 'train_steps_per_second': 4.357, 'total_flos': 1390957231104000.0, 'train_loss': 0.09336362690686091, 'epoch': 3.0})

In [ ]:
# ============================================================
# STEP 1 - INSTALLS
# ============================================================
!pip install transformers datasets scikit-learn torch tqdm evaluate

# ============================================================
# STEP 2 - IMPORTS
# ============================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset
import evaluate
import numpy as np
import torch

print("GPU available:", torch.cuda.is_available())

# ============================================================
# STEP 3 - LOAD DATASET
# ============================================================
dataset = load_dataset("pminervini/HaluEval", "qa_samples", split='data')
print("Dataset loaded:", len(dataset), "samples")

# ============================================================
# STEP 4 - PREPARE INPUT TEXT AND LABELS
# ============================================================
def prepare_input_text(example):
    example['input_text'] = f"Documentation: {example['knowledge']} Question: {example['question']} Answer: {example['answer']}"
    example['labels'] = 1 if example['hallucination'] == 'yes' else 0
    return example

dataset = dataset.map(prepare_input_text)
print("Sample input:", dataset[0]['input_text'])
print("Sample label:", dataset[0]['labels'])

# ============================================================
# STEP 5 - SPLIT DATASET (70/15/15)
# ============================================================
split1 = dataset.train_test_split(test_size=0.3, seed=42)
train_dataset = split1['train']
temp_dataset = split1['test']

split2 = temp_dataset.train_test_split(test_size=0.5, seed=42)
validation_dataset = split2['train']
test_dataset = split2['test']

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(validation_dataset)}")
print(f"Test: {len(test_dataset)}")

# ============================================================
# STEP 6 - TOKENIZATION
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small", use_fast=True)

def tokenize_function(example):
    return tokenizer(
        example['input_text'],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
validation_dataset = validation_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# ============================================================
# STEP 7 - SET FORMAT FOR PYTORCH
# ============================================================
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])
validation_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])

print("Train sample labels:", train_dataset[0]['labels'])

# ============================================================
# STEP 8 - LOAD MODEL
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=2
)

# ============================================================
# STEP 9 - COMPUTE METRICS
# ============================================================
def compute_metrics(eval_pred):
    f1_metric = evaluate.load("f1")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")

    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"]
    precision = precision_metric.compute(predictions=predictions, references=labels, average="binary")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="binary")["recall"]

    return {"f1": f1, "precision": precision, "recall": recall}

# ============================================================
# STEP 10 - TRAINING ARGUMENTS
# ============================================================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50
)

# ============================================================
# STEP 11 - TRAINER
# ============================================================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# ============================================================
# STEP 12 - TRAIN
# ============================================================
trainer.train()

# ============================================================
# STEP 13 - FINAL EVALUATION ON TEST SET
# ============================================================
results = trainer.evaluate(test_dataset)
print("\n" + "="*30)
print("--- DEBERTA FINE-TUNED RESULTS ---")
print("="*30)
print(f"F1:        {results['eval_f1']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall:    {results['eval_recall']:.4f}")
print(f"Loss:      {results['eval_loss']:.4f}")

GPU available: True
Dataset loaded: 10000 samples


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Sample input: Documentation: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA. Question: Which magazine was started first Arthur's Magazine or First for Women? Answer: First for Women was started first.
Sample label: 1
Train: 7000
Validation: 1500
Test: 1500


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


--- DEBERTA FINE-TUNED RESULTS ---
F1:        0.0000
Precision: 0.0000
Recall:    0.0000
Loss:      nan


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# ============================================================
# STEP 1 - INSTALLS
# ============================================================
!pip install transformers datasets scikit-learn torch tqdm evaluate

# ============================================================
# STEP 2 - IMPORTS
# ============================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding # Changed from default_data_collator
from datasets import load_dataset
import evaluate
import numpy as np
import torch
import shutil

print("GPU available:", torch.cuda.is_available())

# ============================================================
# STEP 3 - CLEAR OLD RESULTS
# ============================================================
shutil.rmtree("./results", ignore_errors=True)
print("Cleared old results folder")

# ============================================================
# STEP 4 - LOAD DATASET
# ============================================================
dataset = load_dataset("pminervini/HaluEval", "qa_samples", split='data')
print("Dataset loaded:", len(dataset), "samples")

# ============================================================
# STEP 5 - PREPARE INPUT TEXT AND LABELS
# ============================================================
def prepare_input_text(example):
    example['input_text'] = f"Documentation: {example['knowledge']} Question: {example['question']} Answer: {example['answer']}"
    example['labels'] = 1 if example['hallucination'] == 'yes' else 0
    return example

dataset = dataset.map(prepare_input_text)
print("Sample input:", dataset[0]['input_text'])
print("Sample label:", dataset[0]['labels'])

# ============================================================
# STEP 6 - SPLIT DATASET (70/15/15)
# ============================================================
split1 = dataset.train_test_split(test_size=0.3, seed=42)
train_dataset = split1['train']
temp_dataset = split1['test']

split2 = temp_dataset.train_test_split(test_size=0.5, seed=42)
validation_dataset = split2['train']
test_dataset = split2['test']

print(f"Train:      {len(train_dataset)}")
print(f"Validation: {len(validation_dataset)}")
print(f"Test:       {len(test_dataset)}")

# ============================================================
# STEP 7 - TOKENIZATION
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small", use_fast=True)

def tokenize_function(example):
    return tokenizer(
        example['input_text'],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
validation_dataset = validation_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# ============================================================
# STEP 8 - SET FORMAT FOR PYTORCH
# ============================================================
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])
validation_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])

print("Labels check:", train_dataset[0]['labels'])

# ============================================================
# STEP 9 - LOAD MODEL
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=2
)

# ============================================================
# STEP 10 - COMPUTE METRICS
# ============================================================
def compute_metrics(eval_pred):
    f1_metric = evaluate.load("f1")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")

    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"]
    precision = precision_metric.compute(predictions=predictions, references=labels, average="binary")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="binary")["recall"]

    return {"f1": f1, "precision": precision, "recall": recall}

# ============================================================
# STEP 11 - TRAINING ARGUMENTS
# ============================================================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-6, # Reduced learning rate
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    fp16=False # Explicitly disable float16 training
)

# ============================================================
# STEP 12 - TRAINER
# ============================================================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # Explicitly define data_collator

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    # Removed 'tokenizer=tokenizer' from here
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# ============================================================
# STEP 13 - TRAIN
# ============================================================
trainer.train()

# ============================================================
# STEP 14 - FINAL EVALUATION ON TEST SET
# ============================================================
results = trainer.evaluate(test_dataset)
print("\n" + "="*30)
print("--- DEBERTA FINE-TUNED RESULTS ---")
print("="*30)
print(f"F1:        {results['eval_f1']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall:    {results['eval_recall']:.4f}")
print(f"Loss:      {results['eval_loss']:.4f}")

In [ ]:
# ============================================================
# STEP 1 - INSTALLS
# ============================================================
!pip install transformers datasets scikit-learn torch tqdm evaluate

# ============================================================
# STEP 2 - IMPORTS
# ============================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding # Changed from default_data_collator
from datasets import load_dataset
import evaluate
import numpy as np
import torch
import shutil

print("GPU available:", torch.cuda.is_available())


# ============================================================
# STEP 3 - CLEAR OLD RESULTS
# ============================================================
shutil.rmtree("./results", ignore_errors=True)

# ============================================================
# STEP 4 - LOAD DATASET
# ============================================================
dataset = load_dataset("pminervini/HaluEval", "qa_samples", split='data')
print("Dataset loaded:", len(dataset), "samples")

# ============================================================
# STEP 5 - PREPARE INPUT TEXT AND LABELS
# ============================================================
def prepare_input_text(example):
    example['input_text'] = f"Documentation: {example['knowledge']} Question: {example['question']} Answer: {example['answer']}"
    example['labels'] = 1 if example['hallucination'] == 'yes' else 0
    return example

dataset = dataset.map(prepare_input_text)

# ============================================================
# STEP 6 - SPLIT DATASET (70/15/15)
# ============================================================
split1 = dataset.train_test_split(test_size=0.3, seed=42)
train_dataset = split1['train']
temp_dataset = split1['test']

split2 = temp_dataset.train_test_split(test_size=0.5, seed=42)
validation_dataset = split2['train']
test_dataset = split2['test']

print(f"Train:      {len(train_dataset)}")
print(f"Validation: {len(validation_dataset)}")
print(f"Test:       {len(test_dataset)}")

# Check class distribution
labels = np.array(train_dataset['labels'])
print("Class 0 (not hallucinated):", (labels == 0).sum())
print("Class 1 (hallucinated):", (labels == 1).sum())

# ============================================================
# STEP 7 - TOKENIZATION
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small", use_fast=True)

def tokenize_function(example):
    return tokenizer(
        example['input_text'],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
validation_dataset = validation_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# ============================================================
# STEP 8 - SET FORMAT FOR PYTORCH
# ============================================================
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
validation_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
# ============================================================
# STEP 9 - LOAD MODEL
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=2
)

# ============================================================
# STEP 10 - COMPUTE METRICS
# ============================================================
def compute_metrics(eval_pred):
    f1_metric = evaluate.load("f1")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")

    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"]
    precision = precision_metric.compute(predictions=predictions, references=labels, average="binary")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="binary")["recall"]

    return {"f1": f1, "precision": precision, "recall": recall}

# ============================================================
# STEP 11 - TRAINING ARGUMENTS (lower learning rate)
# ============================================================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    fp16=False # Explicitly disable float16 training
)



# ============================================================
# STEP 12 - TRAINER
# ============================================================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # Explicitly define data_collator

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    # Removed 'processing_class=tokenizer' from here
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

trainer.train()

# ============================================================
# STEP 14 - FINAL EVALUATION ON TEST SET
# ============================================================
results = trainer.evaluate(test_dataset)
print("\n" + "="*30)
print("--- DEBERTA FINE-TUNED RESULTS ---")
print("="*30)
print(f"F1:        {results['eval_f1']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall:    {results['eval_recall']:.4f}")
print(f"Loss:      {results['eval_loss']:.4f}")

GPU available: True
Dataset loaded: 10000 samples
Train:      7000
Validation: 1500
Test:       1500
Class 0 (not hallucinated): 3482
Class 1 (hallucinated): 3518


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.000000,nan,0.000000,0.000000,0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from collections import Counter
Counter(dataset['labels'])

Counter({1: 5010, 0: 4990})

In [ ]:
 pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00


In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [ ]:
# ============================================================
# STEP 1 - INSTALLS
# ============================================================
# !pip install transformers datasets scikit-learn torch tqdm evaluate

# ============================================================
# STEP 2 - IMPORTS
# ============================================================
import numpy as np
import shutil
import torch
from collections import Counter
from torch.nn import CrossEntropyLoss
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
import evaluate

print("GPU available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# STEP 3 - CLEAR OLD RESULTS
# ============================================================
shutil.rmtree("./results", ignore_errors=True)

# ============================================================
# STEP 4 - LOAD DATASET
# ============================================================
dataset = load_dataset("pminervini/HaluEval", "qa_samples", split="data")

# ============================================================
# STEP 5 - PREPARE DATA
# FIX: Removed manual input_text with [SEP] joining.
#      The tokenizer now handles two segments natively.
# ============================================================
def prepare_input(example):
    example["labels"] = 1 if example["hallucination"] == "yes" else 0
    return example

dataset = dataset.map(prepare_input)

label_counts = Counter(dataset["labels"])
print("Label distribution:", label_counts)

# ============================================================
# STEP 6 - SPLIT DATA
# ============================================================
split1 = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split1["train"]
temp_dataset  = split1["test"]

split2 = temp_dataset.train_test_split(test_size=0.5, seed=42)
validation_dataset = split2["train"]
test_dataset       = split2["test"]

print(f"Train: {len(train_dataset)} | Val: {len(validation_dataset)} | Test: {len(test_dataset)}")

# ============================================================
# STEP 7 - TOKENIZER
# FIX: Pass question+answer as Segment A and knowledge as
#      Segment B so the tokenizer sets token_type_ids correctly.
# ============================================================
# The previous loop for checking types was problematic when batched=True
# It's better to ensure robustness within the tokenize function.

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

def tokenize(examples):
    # Ensure inputs are strings, replace None with empty string
    text_a_batch = [f"{q or ''} {a or ''}" for q, a in zip(examples['question'], examples['answer'])]
    text_b_batch = [k or '' for k in examples['knowledge']]
    return tokenizer(
        text_a_batch,
        text_b_batch,
        truncation=True,
        padding="max_length",
        max_length=256,
    )

train_dataset      = train_dataset.map(tokenize, batched=True)
validation_dataset = validation_dataset.map(tokenize, batched=True)
test_dataset       = test_dataset.map(tokenize, batched=True)

# ============================================================
# STEP 8 - FORMAT DATASETS
# FIX: Include token_type_ids (produced by DeBERTa tokenizer).
#      Guard with column_names check in case it's absent.
# ============================================================
cols = [
    c for c in ["input_ids", "attention_mask", "token_type_ids", "labels"]
    if c in train_dataset.column_names
]

train_dataset.set_format("torch", columns=cols)
validation_dataset.set_format("torch", columns=cols)
test_dataset.set_format("torch", columns=cols)

# ============================================================
# STEP 9 - MODEL
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=2,
    ignore_mismatched_sizes=True,
)

# ============================================================
# STEP 10 - CLASS WEIGHTS
# ============================================================
total = sum(label_counts.values())
weights = [total / label_counts[i] for i in range(2)]
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
print("Class weights:", class_weights)

# ============================================================
# STEP 11 - CUSTOM TRAINER
# ============================================================
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs["labels"]
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.dtype))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ============================================================
# STEP 12 - METRICS
# FIX: Added accuracy metric for a fuller picture.
# ============================================================
f1_metric        = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric    = evaluate.load("recall")
accuracy_metric  = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1":        f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
        "precision": precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall":    recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"],
        "accuracy":  accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
    }

# ============================================================
# STEP 13 - TRAINING ARGUMENTS
# FIX: warmup_ratio instead of fixed steps, save_total_limit=2,
#      larger eval batch, bf16 if supported, report_to=none.
# ============================================================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,            # no gradients → can be larger
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,                       # keep only the 2 best checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,                        # safer than fixed warmup_steps
    max_grad_norm=1.0,
    logging_steps=50,
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),      # bf16 is better than fp16 for DeBERTa
    seed=42,
    report_to="none",                         # suppress wandb/tensorboard warnings
)

# ============================================================
# STEP 14 - TRAINER
# ============================================================
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator,
)

# ============================================================
# STEP 15 - TRAIN
# ============================================================
trainer.train()

# ============================================================
# STEP 16 - EVALUATE ON TEST SET
# ============================================================
results = trainer.evaluate(test_dataset)

print("\n==================== FINAL TEST RESULTS ====================")
print(f"F1:        {results['eval_f1']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall:    {results['eval_recall']:.4f}")
print(f"Accuracy:  {results['eval_accuracy']:.4f}")
print(f"Loss:      {results['eval_loss']:.4f}")

# ============================================================
# STEP 17 - PREDICTION DISTRIBUTION CHECK
# ============================================================
preds = trainer.predict(test_dataset)
pred_labels = np.argmax(preds.predictions, axis=1)

unique, counts = np.unique(pred_labels, return_counts=True)
print("\nPrediction distribution:")
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} ({count / len(pred_labels) * 100:.1f}%)")



GPU available: True
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

qa_samples/data-00000-of-00001.parquet:   0%|          | 0.00/3.43M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


==================== FINAL TEST RESULTS ====================
F1:        0.6613
Precision: 0.4940
Recall:    1.0000
Accuracy:  0.4940
Loss:      0.7036

Prediction distribution:
  Class 1: 1000 (100.0%)


In [ ]:
pip download encoder

Saved ./Encoder-1.1-py3-none-any.whl
Successfully downloaded encoder


In [ ]:
pip install evaluate

In [ ]:
# ============================================================
# STEP 1 - INSTALLS
# ============================================================
# !pip install transformers datasets scikit-learn torch tqdm evaluate

# ============================================================
# STEP 2 - IMPORTS
# ============================================================
import numpy as np
import shutil
import torch
from collections import Counter
from torch.nn import CrossEntropyLoss
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
import evaluate

print("GPU available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# STEP 3 - CLEAR OLD RESULTS
# ============================================================
shutil.rmtree("./results", ignore_errors=True)

# ============================================================
# STEP 4 - LOAD DATASET
# ============================================================
dataset = load_dataset("pminervini/HaluEval", "qa_samples", split="data")

# ============================================================
# STEP 5 - PREPARE DATA
# ============================================================
def prepare_input(example):
    example["labels"] = 1 if example["hallucination"] == "yes" else 0
    return example

dataset = dataset.map(prepare_input)

label_counts = Counter(dataset["labels"])
print("Label distribution:", label_counts)

# ============================================================
# STEP 6 - SPLIT DATA
# ============================================================
split1 = dataset.train_test_split(test_size=0.3, seed=42)
train_dataset = split1["train"]
temp_dataset  = split1["test"]

split2 = temp_dataset.train_test_split(test_size=0.5, seed=42)
validation_dataset = split2["train"]
test_dataset       = split2["test"]

print(f"Train: {len(train_dataset)} | Val: {len(validation_dataset)} | Test: {len(test_dataset)}")

# Sanity check — confirm label balance across splits
print("Train:", Counter(train_dataset["labels"]))
print("Val:  ", Counter(validation_dataset["labels"]))
print("Test: ", Counter(test_dataset["labels"]))

# ============================================================
# STEP 7 - TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

def tokenize(examples):
    text_a_batch = [f"{q or ''} {a or ''}" for q, a in zip(examples["question"], examples["answer"])]
    text_b_batch = [k or "" for k in examples["knowledge"]]
    return tokenizer(
        text_a_batch,
        text_b_batch,
        truncation=True,
        padding="max_length",
        max_length=256,
    )

train_dataset      = train_dataset.map(tokenize, batched=True)
validation_dataset = validation_dataset.map(tokenize, batched=True)
test_dataset       = test_dataset.map(tokenize, batched=True)

# ============================================================
# STEP 8 - FORMAT DATASETS
# ============================================================
cols = [
    c for c in ["input_ids", "attention_mask", "token_type_ids", "labels"]
    if c in train_dataset.column_names
]

train_dataset.set_format("torch", columns=cols)
validation_dataset.set_format("torch", columns=cols)
test_dataset.set_format("torch", columns=cols)

# ============================================================
# STEP 9 - MODEL
# FIX: Added dropout to regularize and prevent shortcut learning
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=2,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
    ignore_mismatched_sizes=True,
)

# ============================================================
# STEP 10 - REMOVED CLASS WEIGHTS
# FIX: Class weights were causing the model to always predict
#      Class 1. Replaced with label smoothing in the trainer.
# ============================================================

# ============================================================
# STEP 11 - CUSTOM TRAINER
# FIX: Replaced class weights with label_smoothing=0.1.
#      This prevents overconfident wrong predictions without
#      the instability that inverse-frequency weights caused.
# ============================================================
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs["labels"]
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fct = CrossEntropyLoss(label_smoothing=0.1)  # no class_weights
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ============================================================
# STEP 12 - METRICS
# FIX: Added zero_division=0 to silence UndefinedMetricWarning
# ============================================================
f1_metric        = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric    = evaluate.load("recall")
accuracy_metric  = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1":        f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
        "precision": precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall":    recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"],
        "accuracy":  accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
    }

# ============================================================
# STEP 13 - TRAINING ARGUMENTS
# FIX: Lower LR (5e-6), cosine scheduler, warmup_steps=300,
#      more epochs (5) for stable convergence
# ============================================================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=5e-6,                   # FIX: was 2e-5, too high for DeBERTa-v3
    weight_decay=0.01,
    warmup_steps=300,                     # FIX: replaces deprecated warmup_ratio
    lr_scheduler_type="cosine",           # FIX: cosine decay prevents oscillation
    max_grad_norm=1.0,
    logging_steps=50,
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),
    seed=42,
    report_to="none",
)

# ============================================================
# STEP 14 - TRAINER
# ============================================================
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator,
)

# ============================================================
# STEP 15 - TRAIN
# ============================================================
trainer.train()

# ============================================================
# STEP 16 - EVALUATE ON TEST SET
# ============================================================
results = trainer.evaluate(test_dataset)

print("\n==================== FINAL TEST RESULTS ====================")
print(f"F1:        {results['eval_f1']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall:    {results['eval_recall']:.4f}")
print(f"Accuracy:  {results['eval_accuracy']:.4f}")
print(f"Loss:      {results['eval_loss']:.4f}")

# ============================================================
# STEP 17 - PREDICTION DISTRIBUTION CHECK
# ============================================================
preds = trainer.predict(test_dataset)
pred_labels = np.argmax(preds.predictions, axis=1)

unique, counts = np.unique(pred_labels, return_counts=True)
print("\nPrediction distribution:")
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} ({count / len(pred_labels) * 100:.1f}%)")

GPU available: True
Using device: cuda


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Label distribution: Counter({1: 5010, 0: 4990})
Train: 7000 | Val: 1500 | Test: 1500
Train: Counter({1: 3518, 0: 3482})
Val:   Counter({0: 759, 1: 741})
Test:  Counter({1: 751, 0: 749})


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


==================== FINAL TEST RESULTS ====================
F1:        0.6673
Precision: 0.5007
Recall:    1.0000
Accuracy:  0.5007
Loss:      0.6936

Prediction distribution:
  Class 1: 1500 (100.0%)


In [ ]:
pip install evaluate

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
import shutil
import torch
from collections import Counter
from torch.nn import CrossEntropyLoss
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
import evaluate
dataset = load_dataset("pminervini/HaluEval", "qa_samples", split="data")

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    # Print full classification report
    print("\n" + classification_report(labels, preds, target_names=["Not Hallucinated", "Hallucinated"]))

    return {
        "f1":        f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
        "precision": precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall":    recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"],
        "accuracy":  accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
    }

print(compute_metrics)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs["labels"]
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fct = CrossEntropyLoss(label_smoothing=0.1)  # no class_weights
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

#============================================================
# STEP 14 - TRAINER
# ============================================================
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator,
)

from sklearn.metrics import classification_report

preds = trainer.predict(test_dataset)
pred_labels = np.argmax(preds.predictions, axis=1)
true_labels = preds.label_ids

print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(
    true_labels,
    pred_labels,
    target_names=["Not Hallucinated", "Hallucinated"]
))

<function compute_metrics at 0x7f9099b227a0>


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



                  precision    recall  f1-score   support

Not Hallucinated       0.00      0.00      0.00       749
    Hallucinated       0.50      1.00      0.67       751

        accuracy                           0.50      1500
       macro avg       0.25      0.50      0.33      1500
    weighted avg       0.25      0.50      0.33      1500


========== CLASSIFICATION REPORT ==========
                  precision    recall  f1-score   support

Not Hallucinated       0.00      0.00      0.00       749
    Hallucinated       0.50      1.00      0.67       751

        accuracy                           0.50      1500
       macro avg       0.25      0.50      0.33      1500
    weighted avg       0.25      0.50      0.33      1500



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
